# Manage Delta lake

In [ ]:
from deltalake import DeltaTable, write_deltalake
import pyarrow as pa 
from eerssa.utils import load_r2_credentials
from datetime import datetime

def force_string_columns(table: pa.Table, columns: list[str]) -> pa.Table:
    schema = table.schema
    for col in columns:
        idx = schema.get_field_index(col)
        schema = schema.set(idx, pa.field(col, pa.string()))
    return table.cast(schema)

# Conectar con DELTA LAKE TABLE (Cloudflare R2)
# DELTA_TABLE_PATH_ON_HOST
R2_BUCKET = "delta-v30"
CREDS_PATH = "secrets/r2_credentials.json"
creds = load_r2_credentials(CREDS_PATH)
R2_ENDPOINT = f"https://{creds['account_id']}.r2.cloudflarestorage.com"
table_path =  f"s3://{R2_BUCKET}/delta_v30"

storage_options = {
    "AWS_ENDPOINT_URL": R2_ENDPOINT,
    "AWS_ACCESS_KEY_ID": creds["access_key"],
    "AWS_SECRET_ACCESS_KEY": creds["secret_key"],
    "AWS_REGION": "auto",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

version_deltalake = set()


## Initial setup

In [9]:

dt = DeltaTable(table_path, storage_options=storage_options)
df = dt.to_pandas()
version_deltalake.add(dt.version())
print(f"Current version: {dt.version()}")
print(f"Unique versions seen this session: {sorted(version_deltalake)}")
print(f"Number of unique versions: {len(version_deltalake)}")


Current version: 455
Unique versions seen this session: [455]
Number of unique versions: 1


## Deltalake History

In [92]:
print("\n--- Table History ---")
history = dt.history(limit = 10)
for commit in history:
    print(
        f"Version: {commit['version']}, "
        f"Timestamp: {datetime.fromtimestamp(commit['timestamp']/1000)}, "
        f"Operation: {commit['operation']}"
    )



--- Table History ---
Version: 457, Timestamp: 2026-07-22 10:55:20.742000, Operation: MERGE
Version: 456, Timestamp: 2026-07-22 10:16:08.936000, Operation: MERGE
Version: 455, Timestamp: 2026-07-22 09:20:03.841000, Operation: WRITE
Version: 454, Timestamp: 2026-07-22 09:18:58.023000, Operation: DELETE
Version: 453, Timestamp: 2026-07-22 08:58:14.994000, Operation: WRITE
Version: 452, Timestamp: 2026-07-22 08:57:51.854000, Operation: WRITE
Version: 451, Timestamp: 2026-07-22 08:57:21.035000, Operation: WRITE
Version: 450, Timestamp: 2026-07-22 08:56:57.360000, Operation: WRITE
Version: 449, Timestamp: 2026-07-22 08:56:36.097000, Operation: WRITE
Version: 448, Timestamp: 2026-07-22 08:56:07.802000, Operation: WRITE


## Eliminar una OT de Deltalake

In [6]:
dt.delete("id_ot = 81013")

{'num_added_files': 1,
 'num_removed_files': 1,
 'num_deleted_rows': 131072,
 'num_copied_rows': 39754,
 'execution_time_ms': 4577,
 'scan_time_ms': 983,
 'rewrite_time_ms': 0}

# ✍️ Guardar cambios en Deltalake

> Se guardan los cambios de `df`

In [163]:
# Escribir cambios en Delta-lake  desde filered_df

# --- Force Fecha to plain pa.string() before merge ---
source_table = pa.Table.from_pandas(df, preserve_index=False)
source_table = force_string_columns(source_table, ["Fecha"])
# ------------------------------------------------------

try:

    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"


    (dt.merge(
                    source=source_table,
                    predicate=unique_key_predicate,
                    source_alias="source",
                    target_alias="target"
                )
                .when_matched_update_all()  # Rule 1: If an activity exists, update it.
                .when_not_matched_insert_all()  # Rule 2: If it's a new activity, insert it.
                .execute()
    )
    print("✅ **Successfully saved changes to Delta Lake!**\n\n")

    # Imprimir los resultados
    dt = DeltaTable(table_path, storage_options=storage_options)
    df = dt.to_pandas()
    version_deltalake.add(dt.version())
    print(f"Current version: {dt.version()}")
    print(f"Unique versions seen this session: {sorted(version_deltalake)}")
    print(f"Number of unique versions: {len(version_deltalake)}")

except Exception as e:
    print(f"❌ **Error saving to Delta Lake:** {e}")


✅ **Successfully saved changes to Delta Lake!**


Current version: 462
Unique versions seen this session: [455, 456, 457, 458, 459, 460, 461, 462]
Number of unique versions: 8


## Visualizar OT en base al `id_ot`

In [11]:
_ot_to_view = "177422"
_view_ot = df.query( f"id_ot == {_ot_to_view}" ).sort_values(by='Item')
_view_ot

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,Year,Iniciales
56357,1,informativa,FERIADO POR EL 24 DE MAYO BATALLA DE PICHINCHA...,INFO,·,No,No,No,·,·,...,0.016667,QUIROGA ORDONEZ CARLOS HERNAN,1,No,R-142,Guayzimi,177422,OT [07] Cuadrilla Guayzimi 2026-05-25 (038) CQ...,2026,.
53888,3,se_labora,"SE LABORA CQ, JL, DE:.....",LABORA,·,No,No,No,·,·,...,0.016667,QUIROGA ORDONEZ CARLOS HERNAN,1,No,R-142,Guayzimi,177422,OT [07] Cuadrilla Guayzimi 2026-05-25 (038) CQ...,2026,.


## Cambio de TIPO de trabajo mal escrita

Hay casos en los que las Cuentas se guardan con espacios al final. por ejemplo `'CORRECTIVO '` y también `'CORRECTIAS '`
Se ha corregido este error a partir de la version. `gestion 5.1` sin embargo es necesario corregir los datos que ya 
constan en el dataset. El Siguiente código identifica y corrige estas variaciones para tener valores unificados del tipo
de actividad

In [162]:
df['Tipo'].unique()

array(['PREVENTIVO', 'TRANSPORTE', '·', 'CORRECTIVO', 'RUTINARIA',
       'EXPANSION', 'PREDICTIVO', 'LUNCH', 'PREVENTIVO ', 'LABORA'],
      dtype=object)

In [ ]:
"""
    El objetivo es identificar el tipo de actividad con un espacio al final 
    para ser reemplazadas por un mismo valor uniforme
    
    FILTROS:
    * Identificamos el texto: 'CORRECTIVAS '  "CORRECTIVO "

    ACTGIS ->  PREDICTIVO
    
"""
cuenta_erronea = 'CORRECTIVAS'
cuenta_correcta = "CORRECTIVO"

mask = df['Tipo'] == cuenta_erronea
#mask = df['Tipo'].str.match(r'202', na=False)
filtered_df = df[mask]
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,Year,Iniciales
1253,1,informativa,ada\nSe elabora OT. Se coordina trabajos con e...,Program,·,No,No,No,202\n06:3,·,...,390.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Yantzaza.,177526,27 0T MIERCOLES 2026-signed-signed-signed-sign...,2026,.
1642,1,informativa,ada\nSe elabora OT. Se coordina trabajos con e...,Program,·,No,No,No,202\n08:0,·,...,500.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Chuchumbleza.,177185,21 OT JUEVES MAYO 2026-signed-signed-signed-si...,2026,.
3018,1,informativa,ada\nSe elabora OT. Se coordina trabajos con e...,Program,·,No,No,No,202\n08:0,·,...,540.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Pangui -Gualaquiza.,176717,14 0T JUEVES MAYO 2026-signed-signed-signed-si...,2026,.
4705,1,informativa,ada\nSe elabora OT. Se coordina trabajos con e...,Program,·,No,No,No,202\n08:0,·,...,510.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,LAS PEÑAS - CHUCHUMBLEZA.,177097,20 OT MIERCOLES MAYO 2026-signed-signed-signed...,2026,.
6666,1,informativa,ada\nSe elabora OT. Se coordina trabajos con c...,Program,·,No,No,No,202\n18:0,·,...,495.0,AGURTO BARRAGAN LUIS DARWIN,1,Si,2-45,YANTZAZA,176869,17 OT DOMINGO MAYO 2026-signed-signed-signed-s...,2026,.
7094,2,informativa,ada\nSubestación Yantzaza: Se coordina trabajo...,Program,·,No,No,No,202\n07:0,·,...,480.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Los Encuentros.,177611,28 OT JUEVES MAYO 2026-signed-signed-signed-si...,2026,.
7158,3,informativa,ada\nLoja: Bodega General: Se coordina con el ...,Program,·,No,No,No,202\n09:0,·,...,270.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Loja canasta 2-146 y Camioneta 2-45 Yantzaza.,176600,13 0T MIERCOLES MAYO 2026-signed-signed-signed...,2026,.
8118,1,informativa,ada\nSe elabora OT. Se coordina trabajos con e...,Program,·,No,No,No,202\n08:0,·,...,540.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Yantzaza- Los Encuentros.,176430,11 0T LUNES MAYO 2026-signed-signed-signed-sig...,2026,.
8888,1,informativa,ada\nElebarar OT. Se coordina trabajos con el ...,Program,·,No,No,No,202\n08:0,·,...,510.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Gualaquiza.,176792,15 OT VIERNES MAYO 2026-signed-signed-signed-s...,2026,.
9223,1,informativa,ada\nSen elabora OT. Se coordina trabajos con ...,Program,·,No,No,No,202\n08:0,·,...,510.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,El Pangui.,176307,08 OT VIERNES MAYO 2026-signed-signed-signed-s...,2026,.


In [161]:
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Tipo'] = cuenta_correcta
print(f"✅ Se ha actualizado el Dataframe, se reemplazo |{cuenta_erronea}| y se escribió |{cuenta_correcta}|")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 29 filas por modificar.
✅ Se ha actualizado el Dataframe, se reemplazo |ANSPORTE| y se escribió |CORRECTIVO|


## Identificar cuantas de tipo "Servicios_Ocasionales"

In [129]:
df['Cuenta'].unique()


array(['REDES', 'informativa', 'transporte', 'lunch', 'se_labora',
       'ACOMETIDAS', '?', 'ALUMBRADO', 'MEDIDORES', 'Servicios_Ocasional',
       'Nuevos_Servici', ' Servicios_Ocasional', 'SUBESTACION'],
      dtype=object)

In [137]:
"""
    El objetivo es identificar el tipo de actividad (Cuenta) que se encuentran como desconocidas (?) 
    pero que se pueden atribuir como cuenta de tipo "Servicios_Ocasionales
    
    FILTROS:
    * Debe contar con un Alimentador identificado
    * Cuenta: Desconocida (?)
    * Tipo: RUTINARIA
    * Que en el texto (Evento) contengan las palabras, 's/o' o 'ocasional'

    Estas filas son de Tipo = Servicios_Ocasionales, 

    primero generamos una máscara para identificarlas, luego aplicamos el cambio de cuenta,
    verificamos y guardamos en el Dataset.
"""
cond1 = df['Alimentador'] != "·"
cond2 = df['Tipo'].str.contains("RUTINARIA", regex=False, na=False, case=False)
cond3 = df['Cuenta'] == '?'
cond4 = df['Evento'].str.contains("ocasional", regex=False, na=False, case=False)

# Combine the conditions with the "&" (AND) operator to create the final boolean mask
mask = cond1 & cond2 & cond3 & cond4


# Apply the mask to the DataFrame to get the filtered result
filtered_df = df[mask]

# Display the filtered DataFrame
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,Year,Iniciales
352,8,?,"Nueva Tarqui, estructura #185529, instalacion ...",PROG,Bomboiza,No,No,No,RUTINARIA,·,...,30.0,BUELE UYAGUARI CESAR CRISTIAN,0,No,R-12,"Guabi, Gualaquiza, Chumpias, Guayusal",120060,OT [24] Agencia Gualaquiza 2023-12-07 (058) CB...,2023,.
520,8,?,En Zumbi se instala servicio ocasional orden #...,PROG,Paquisha,No,No,No,RUTINARIA,·,...,30.0,POMA GUAMAN LUIS ANTONIO,1,No,4-100,UNION LOJANA - ZUMBI - YANTZAZA - CHINAPINTZA.-,99595,OT [22] Agencia Yantzaza 2023-01-06 (053) LP.pdf,2023,.
704,10,?,"En Zumbi, estructura 181141, se retira servici...",PROG,Paquisha,No,No,No,RUTINARIA,·,...,25.0,CHAMBA CANGO PEDRO ROSALINO,1,No,4-100,"YANTZAZA, ZUMBI.",105379,OT [22] Agencia Yantzaza 2023-04-14 (052) PCH.pdf,2023,.
792,15,?,En el Pindal se retira S:Ocasional `estructura...,PROG,Los Encuentros,No,No,No,RUTINARIA,·,...,20.0,CUADRADO LEON RAUL EFREN,1,No,4-36,"Yantzaza, Los Hachos, Miraflores, Chicaña,Muc...",119397,OT [22] Agencia Yantzaza 2023-11-28 (051) RCL.pdf,2023,.
895,3,?,"En Guayzimi sector el Parque Central,estructur...",PROG,Paquisha,No,No,No,RUTINARIA,·,...,60.0,CHAMBA CANGO PEDRO ROSALINO,2,No,4-100,"GUAYZIMI,",119051,OT [22] Agencia Yantzaza 2023-11-22 (052) PCH.pdf,2023,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296883,15,?,En Yantzaza en el Terminal se instala S.Ocasio...,PROG,Yantzaza III,No,No,No,RUTINARIA,·,...,30.0,CUADRADO LEON RAUL EFREN,2,No,2-53,"Yantzaza,Los Hachos ,Playa Rica ,Panguintza",17726,OT [22] Agencia Yantzaza 2018-10-11 (051) RCL.pdf,2018,.
296913,15,?,"Gualaquiza, Cdla Perla de la Amazonia estructu...",PROG,Gualaquiza,No,No,No,RUTINARIA,·,...,35.0,BUELE UYAGUARI CESAR CRISTIAN,0,No,R-102,"Alto Yutui, Belen, Gualaquiza",16533,OT [24] Agencia Gualaquiza 2018-09-12 (058) CB...,2018,.
297463,12,?,"Bomboiza, estructuranro037229 instalacion serv...",PROG,Gualaquiza,No,No,No,RUTINARIA,·,...,30.0,ROMERO ARIAS KLEVER ALEJANDRO,1,No,R-101,"Bomboiza, Guabi Bajo, Gualaquiza",21157,OT [24] Agencia Gualaquiza 2018-12-19 (0) KR.pdf,2018,.
297526,7,?,"En el Progreso, se realiza la instalación de u...",PROG,Yacuambi,No,No,No,RUTINARIA,·,...,30.0,RIOS RIOS FRANCISCO FERNANDO,1,No,R-18,"Zamora, Napintza, Cusuntza",21182,OT [01] Cuadrilla Zamora 2018-12-21 (006) FR.pdf,2018,.


In [138]:
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Cuenta'] = 'Servicio_Ocasional'
print("✅ Se ha actualizado el Dataframe")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 761 filas por modificar.
✅ Se ha actualizado el Dataframe


## Identificar cuantas de tipo "Acometidas" y "Medidores"

In [144]:
"""
    El objetivo es identificar el tipo de actividad (Cuenta) que se encuentran como desconocidas (?) 
    pero que se pueden atribuir como cuenta de tipo "Servicios_Ocasionales
    
    FILTROS:
    * Debe contar con un Alimentador identificado
    * Cuenta: Desconocida (?)
    * Tipo: RUTINARIA
    * Que en el texto (Evento) contengan las palabras, 'acometida' o 'medidor'

    Estas filas son de Tipo = Servicios_Ocasionales, 

    primero generamos una máscara para identificarlas, luego aplicamos el cambio de cuenta,
    verificamos y guardamos en el Dataset.
"""
cond1 = df['Alimentador'] != "·"
cond2 = df['Cuenta'] == "?" 
cond3 = df['Evento'].str.contains("medidores", regex=False, na=False, case=False)

# Combine the conditions with the "&" (AND) operator to create the final boolean mask
mask = cond1 & cond2 & cond3

# Apply the mask to the DataFrame to get the filtered result
filtered_df = df[mask]

# Display the filtered DataFrame
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,Year,Iniciales
33,2,?,"Yantzaza, se revisa lecturas de medidores # 34...",PROG,Yantzaza III,No,No,No,PREDICTIVO,·,...,75.0,POMA GUAMAN LUIS ANTONIO,1,No,4-100,LOS ENCUENTROS - SAN ANTONIO - LOS ALMENDROS -...,112682,OT [22] Agencia Yantzaza 2023-08-09 (053) LP.pdf,2023,.
834,6,?,En Ssan Francisco se instala orden 4913669 med...,PROG,Yantzaza III,No,No,No,RUTINARIA,·,...,90.0,CUADRADO LEON RAUL EFREN,0,No,4-36,Yantzaza,117786,OT [22] Agencia Yantzaza 2023-10-31 (051) RCL.pdf,2023,.
1783,9,?,Traslado a Gualaquiza se instala 5 medidores 2...,PROG,Gualaquiza,No,No,No,EXPANSION,·,...,170.0,GUZMAN BARROS MARCO FERNANDO,1,No,R-104,"Gualaquiza, Guayusal, San Jose de Piunts, La ...",105544,OT [24] Agencia Gualaquiza 2023-04-17 (042) FG...,2023,.
2180,7,?,En 13 de Junio se instala N.S. de proyecto tri...,PROG,La Saquea,No,No,No,RUTINARIA,·,...,75.0,CUADRADO LEON RAUL EFREN,0,No,2-09,"Yantzaza, 13 de Junio, Zumbi, Panguintza",117014,OT [22] Agencia Yantzaza 2023-10-19 (051) RCL.pdf,2023,.
2634,4,?,Gualaquiza en el Recinto Ferial se realiza ins...,PROG,Gualaquiza,No,No,No,EXPANSION,·,...,32.0,GUZMAN BARROS MARCO FERNANDO,1,No,R-149,"Gualaquiza, Sevilla, Ideal.",115169,OT [24] Agencia Gualaquiza 2023-09-19 (042) FG...,2023,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297055,9,?,En El Pangui se realiza la instalación de medi...,PROG,El Pangui,No,No,No,EXPANSION,·,...,90.0,CARTUCHE SILVA GERARDO PATRICIO,1,No,4-118,"Zamora chinchipe, El Pangui, El Guismi, San Ca...",137422,OT [23] Agencia El Pangui 2024-09-12 (059) GC.pdf,2024,.
297056,12,?,En la Av Héroes de Paquisha y Manuelita Cañiza...,PROG,Zamora I,No,No,No,EXPANSION,·,...,60.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"ZAMORA, TIMBARA",124941,OT [21] Agencia Zamora 2024-02-27 (047) GJ.pdf,2024,.
297293,6,?,En el Pangui se realiza realiza la instalación...,PROG,El Pangui,No,No,No,EXPANSION,·,...,75.0,GRANDA MORALES DANIEL ALEJANDRO,1,No,4-118,"El Pangui, Guismi, Tundayme, Machinza, San Rou...",135876,OT [23] Agencia El Pangui 2024-08-20 (033) DG.pdf,2024,.
298666,4,?,En el Barrio pío Jaramillo se realiza inspecci...,PROG,Zamora II,No,No,No,EXPANSION,·,...,45.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"ZAMORA, TIMBARA,",129651,OT [21] Agencia Zamora 2024-05-09 (047) GJ.pdf,2024,.


In [145]:
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Cuenta'] = "MEDIDORES"
print("✅ Se ha actualizado el Dataframe")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 1084 filas por modificar.
✅ Se ha actualizado el Dataframe


## Novedades con la Fecha Final

In [ ]:
"""
    El objetivo es identificar cuando las actividades finalizan a las 23:59:00 
    pero en lugar de eso se escribe: 00:00:00
    
    FILTROS:
    * Hora final es 00:00:00
    * Hora Inicial ~NO ES~ 00:00:00
    
    Se deben cambiar a : 23:59:00 
"""

cond1 = df['FinEvento'].str.endswith("00:00:00", na=False)
cond2 = df['InicioEvento'].str.endswith("00:00:00", na=False)

#cond2 = df['Evento'].str.contains("Fest", regex=False, na=False, case=False)

# Combine the conditions with the "&" (AND) operator to create the final boolean mask
mask = cond1 & ~cond2

# Apply the mask to the DataFrame to get the filtered result
filtered_df = df[mask]

# Display the filtered DataFrame
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,Year,Iniciales
1300011,3,lunch,Lunch en La Y del Guismi.,ALIMEN,·,No,No,No,LUNCH,·,...,-720.0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"El Pangui, Gualaquiza",169771,OT [02] Alumbrado Zamora 2026-01-29 (012) LM.pdf,2026,.


### Muestra fechas de actividades con posible conflicto

In [ ]:
# Fecha FinEvento Termina en 00:00:00
cond1 = df['FinEvento'].str.endswith("00:00:00", na=False)

# Condiciones en negativo
cond2 = df['Alimentador'].str.contains('·', na=False)
cond3 = df['Evento'].str.contains('Fest', na=False, case=False)
cond4 = df['Evento'].str.contains('Feriado', na=False, case=False)
cond5 = df['Evento'].str.contains('Resolucion', na=False, case=False)
cond6 = df['Evento'].str.contains('labora', na=False, case=False)

# Combine the conditions with the "&" (AND) operator to create the final boolean mask
mask = cond1 & ~cond2 & ~cond3 & ~cond4 & ~cond5 & ~cond6

# Apply the mask to the DataFrame to get the filtered result
filtered_df = df[mask]

# Display the filtered DataFrame
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,Year,Iniciales


In [178]:
df.query(f"Alimentador != '·'").sort_values(by="Duracion",ascending=True).head(10)

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo,Year,Iniciales
1286848,2,REDES,ada\nCumbaraza: estructura # 280044 se realiza...,Program,EXP,No,No,No,EXPANSION,·,...,-900.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Bellavista - Los Encuentros.,176149,06 OT MIERCOLES MAYO 2026-signed-signed-signed...,2026,.
1298592,2,REDES,ada\nCentro de Yantzaza se realiza termografía...,Program,PRE,No,No,No,PREVENTIVO,·,...,-900.0,AGURTO BARRAGAN LUIS DARWIN,1,Si,2-45,YANTZAZA,176869,17 OT DOMINGO MAYO 2026-signed-signed-signed-s...,2026,.
1293495,9,?,rte\nEl Pangui - Subestación Yantzaza - Agenci...,Transpo,TRA,No,No,No,TRANSPORTE,·,...,-760.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Los Encuentros - El Pangui.,176228,07 0T JUEVES MAYO 2026-signed-signed-signed-si...,2026,.
1307518,6,REDES,ada\nRecta del Pangui: estructura # 37525 insp...,Program,EXP,No,No,No,EXPANSION,·,...,-630.0,AGURTO BARRAGAN LUIS DARWIN,5,No,2-45,Los Encuentros - El Pangui.,176228,07 0T JUEVES MAYO 2026-signed-signed-signed-si...,2026,.
1305562,7,ALUMBRADO,"Inspección, Muchime estructura 161985 luminari...",PROG,Los Encuentros,No,No,No,CORRECTIVO,·,...,-585.0,MONTANO AGUILERA JOFFRE PATRICIO,3,No,2-123,Yantzaza,177019,OT [no] Cuadrilla Loja 2026-05-19 (0) JMA.pdf,2026,.
1306415,2,ALUMBRADO,"R.-Pangui,1lum150w,apagada,pst 127974 se cambi...",PROG,El Pangui,No,No,No,CORRECTIVO,·,...,-555.0,ORELLANA BRAVO JORGE LUIS,3,No,2-30,EL PANGUI,169361,OT [no] Cuadrilla Loja 2026-01-22 (0) JOB.pdf,2026,.
1308890,6,ACOMETIDAS,En San Javier se realiza el cambio y empalme d...,NO PROG,La Saquea,No,No,No,CORRECTIVO,·,...,-60.0,"DG, JJ",1,Si,R-184,"Zamora Chinchipe, Paquisha, Bellabista, San Ja...",177324,sc_pdf_20260528073009_371_pdfreport_ordenesTra...,2026,.
1288017,3,MEDIDORES,"Portón estructura 128404 se revisa medidor, se...",NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,-60.0,GUZMAN BARROS MARCO FERNANDO,1,No,2-111,"Gualaquiza, San José de Piunts, La Esperanza.",170310,OT [09] Cuadrilla Gualaquiza 2026-02-05 (045) ...,2026,.
1298975,5,ALUMBRADO,"INC No. 1103888976, Atendido, luminaria de 150...",PROG,La Saquea,No,No,No,CORRECTIVO,·,...,-45.0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"Zamora, Yantzaza, Mercadillo",169907,OT [02] Alumbrado Zamora 2026-02-02 (012) LM.pdf,2026,.
1289801,9,ALUMBRADO,Inspección realizada para repotenciar. Existen...,PROG,Zamora II,No,No,No,PREVENTIVO,·,...,-45.0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,Zamora,180405,OT [02] Alumbrado Zamora 2026-07-10 (012) LM.pdf,2026,.


# ⚠️🚨 RESTAURAR DELTALAKE ⚠️🚨

Utilizar con cuidado!!!!  Por seguridad, esta celda se guarda como Markdown, para utilizar se debe convertir a Python

target_version = 456

# --- Restore using a version number ---
dt.restore(target_version)
        
print(f"✅ Restauración exitosa! La tabla se encuentra en la version {dt.version()}.")

df = dt.to_pandas()
version_deltalake.add(dt.version())
print(f"Current version: {dt.version()}")
print(f"Unique versions seen this session: {sorted(version_deltalake)}")
print(f"Number of unique versions: {len(version_deltalake)}")

## Optimización y Aspirado

In [ ]:
dt.optimize.compact()

In [ ]:
dt.vacuum(retention_hours=100, enforce_retention_duration=False, dry_run=True)  # Cambiar Dry-Run a False para ejecutar aspirado: ⚠️